# ⚖️ Direct Preference Optimization (DPO) for Educational RAG
### Aligning `Meta-Llama-3-8B-Instruct` for Factual Grounding & Hallucination Refusal
### Frameworks: `trl.DPOTrainer`, `peft`, `transformers`, `bitsandbytes`

---
### 📌 Overview
This notebook trains an educational LLM using **Direct Preference Optimization (DPO)** via TRL's `DPOTrainer`.
- **Why DPO?**: While SFT teaches the model *how to speak*, DPO teaches the model *what to prefer*:
  - **Chosen Response**: Strict factual adherence, accurate citations, and proper refusal when context is missing.
  - **Rejected Response**: Hallucinations, external knowledge injection, or missing citations.
- **Hardware Requirement**: Runs on a **Google Colab T4 GPU (15 GB VRAM)** with 4-bit QLoRA.
- **Reference Model**: When using PEFT/LoRA, `DPOTrainer` uses the base model weights with LoRA disabled as the implicit frozen reference policy!

In [ ]:
# Step 1: Install Required Libraries in Colab
!pip install -q --upgrade pip
!pip install -q transformers>=4.40.0 trl>=0.8.6 peft>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.29.0 datasets>=2.18.0

In [ ]:
# Step 2: Verify GPU Acceleration
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")
else:
    raise RuntimeError("Please enable GPU acceleration in Colab: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Step 3: Hugging Face Authentication
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face via Colab Secret!")
except Exception:
    login()

In [ ]:
# Step 4: Create Educational Preference Dataset (Chosen vs. Rejected)
import json

dpo_data = [
    {
        # Example 1: Grounded answer vs. Hallucinated answer
        "prompt": "<|start_header_id|>system<|end_header_id|>\nYou are an expert educational AI tutor. Answer questions factually and strictly based on the provided context with citations.<|eot_id|><|start_header_id|>user<|end_header_id|>\nContext Chunks:\n[1] Isaac Newton published the Principia in 1687, presenting the three laws of motion.\n\nQuestion: When was the Principia published?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n",
        "chosen": "According to the provided context, the Principia was published in 1687 by Isaac Newton [1].",
        "rejected": "The Principia was written by Albert Einstein in 1905, establishing modern general relativity and quantum mechanics."
    },
    {
        # Example 2: Out-of-Domain Query -> Proper Refusal vs. Speculative Hallucination
        "prompt": "<|start_header_id|>system<|end_header_id|>\nYou are an expert educational AI tutor. If the context does not contain the answer, you must decline.<|eot_id|><|start_header_id|>user<|end_header_id|>\nContext Chunks:\n[1] Newton's second law is defined by the formula F = ma.\n\nQuestion: What is the capital city of planet Neptune?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n",
        "chosen": "I am sorry, but the provided context does not contain sufficient educational information to answer this question.",
        "rejected": "The capital city of Neptune is Triton City, where methane clouds provide hydroelectric power to ice habitats."
    },
    {
        # Example 3: Precise Citation vs. Missing Citations
        "prompt": "<|start_header_id|>system<|end_header_id|>\nYou are an expert educational AI tutor. You must provide numeric chunk citations [1].<|eot_id|><|start_header_id|>user<|end_header_id|>\nContext Chunks:\n[1] Newton's Third Law states that every action has an equal and opposite reaction.\n\nQuestion: What does Newton's Third Law state?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n",
        "chosen": "Newton's Third Law states that for every action, there is an equal and opposite reaction [1].",
        "rejected": "Forces are basically equal and opposite whenever things collide."
    }
]

with open("dpo_dataset.jsonl", "w", encoding="utf-8") as f:
    for entry in dpo_data:
        f.write(json.dumps(entry) + "\n")

print(f"Created dpo_dataset.jsonl with {len(dpo_data)} preference pairs.")

In [ ]:
# Step 5: Configure 4-bit Quantization and Load Base Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading Tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading 4-bit Quantized Model {model_id}...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
print("Base model ready for DPO fine-tuning.")

In [ ]:
# Step 6: Configure LoRA for DPO
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 7: Load Preference Dataset
from datasets import load_dataset

dpo_dataset = load_dataset("json", data_files="dpo_dataset.jsonl", split="train")
print(f"Loaded {len(dpo_dataset)} preference pairs.")
print("Sample prompt:", dpo_dataset[0]["prompt"][:120])
print("Sample chosen:", dpo_dataset[0]["chosen"])
print("Sample rejected:", dpo_dataset[0]["rejected"])

In [ ]:
# Step 8: Configure and Run DPOTrainer
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./dpo_output",
    beta=0.1,                            # Regularization parameter: higher = stays closer to base policy
    learning_rate=5e-6,                  # DPO uses smaller learning rate than SFT
    lr_scheduler_type="cosine",
    per_device_train_batch_size=1,       # Low batch size to preserve Colab VRAM
    gradient_accumulation_steps=4,
    max_length=512,
    max_prompt_length=256,
    num_train_epochs=3,
    optim="paged_adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,                      # For PEFT, DPOTrainer disables LoRA adapters for reference policy
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
)

print("Starting Direct Preference Optimization (DPO)...")
dpo_trainer.train()
print("DPO Training Completed Successfully!")

In [ ]:
# Step 9: Save DPO-Aligned Adapter
dpo_save_dir = "./educational_dpo_adapter"

dpo_trainer.model.save_pretrained(dpo_save_dir)
tokenizer.save_pretrained(dpo_save_dir)

print(f"DPO-aligned LoRA Adapter saved to '{dpo_save_dir}'!")
!ls -lh {dpo_save_dir}